# Stellarator wall

A first wall around a stellarator plasma as a 3D surface, and where straight lines meet it:
neutral-beam markers entering the plasma, and their shine-through reaching the wall.
`FourierSurface` holds a toroidal surface in VMEC's and ConStellaration's `SurfaceRZFourier`
layout, `FourierSurface.offset` builds a wall at a given distance outside a plasma boundary,
and `first_hit` finds the first crossing of many segments at once.

The search runs on a triangulation of the surface with a compiled kernel (a grid of cells,
Möller–Trumbore in the cells each segment passes), and every hit is then moved onto the exact
Fourier surface by Newton steps.

## A W7-X-shaped stand-in

`simgeometry.data.w7x_boundary()` is the plasma boundary of DESC's W7-X example equilibrium
(standard configuration, five field periods). W7-X's real first wall (heat shield, baffles,
divertor targets, NBI armour) is IPP's CAD and not public, so `w7x_like_wall` stands in for it:
the boundary moved outward along its normal by a uniform gap of 10 cm. The gap is a round
number, not a W7-X value, and there are no ports. A real wall would be loaded as a triangle mesh
with `TriangleMesh.read`, with the same queries.

In [ ]:
import numpy as np

from simgeometry.data import w7x_boundary, w7x_like_wall

plasma = w7x_boundary()
wall = w7x_like_wall(gap=0.10)
print(
    f"plasma: M = {plasma.n_poloidal}, N = {plasma.n_toroidal}, V = {plasma.volume():.2f} m^3"
)
print(
    f"wall:   M = {wall.n_poloidal}, N = {wall.n_toroidal}, V = {wall.volume():.2f} m^3"
)
print(f"offset misfit: {1e3 * wall.offset_residual:.2f} mm")
print(
    f"{wall.mesh.n_faces} triangles, within {1e3 * wall.triangulation_error:.1f} mm "
    "of the surface"
)

`offset` moves every point of a fine grid along the outward normal and fits the moved
surface again as a Fourier series, with the fewest modes that keep the misfit below a
hundredth of the gap. `closest` (Newton steps on the distance) checks the result: the gap
holds to the misfit everywhere.

In [ ]:
theta, phi = np.meshgrid(
    np.linspace(0, 2 * np.pi, 96, endpoint=False),
    np.linspace(0, 2 * np.pi / 5, 24, endpoint=False),
)
_, _, gap = plasma.closest(wall.point(theta, phi))
print(f"gap from {1e3 * gap.min():.2f} mm to {1e3 * gap.max():.2f} mm")
assert np.allclose(gap, 0.10, atol=2e-3)

## Cross-sections

The plasma boundary and the wall at three toroidal angles of a field period: the bean-shaped
section at $\varphi = 0$, the triangular one at half a period, and one between.

In [ ]:
import matplotlib.pyplot as plt

angles = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(1, 3, figsize=(10, 4), sharey=True)
for ax, fraction in zip(axes, (0.0, 0.25, 0.5)):
    phi0 = fraction * 2 * np.pi / 5
    for surface, label in ((plasma, "plasma boundary"), (wall, "wall (stand-in)")):
        r, z = surface.rz(angles, phi0)
        ax.plot(r, z, label=label)
    ax.set_aspect("equal")
    ax.set_title(f"$\\varphi$ = {np.degrees(phi0):.0f}°")
    ax.set_xlabel("R [m]")
axes[0].set_ylabel("Z [m]")
axes[0].legend(loc="lower left", fontsize=8)
plt.show()

## Lines through the vessel

A fan of horizontal lines starts 2 cm inside the wall at the outboard midplane, as from a
port, and aims at tangency radii from 4.6 m to 6 m. On the plasma, `first_hit` gives where
each line enters it; on the wall, where it leaves the vessel: the spot that the shine-through
heats. Points and segments are `(..., 3)` arrays, as freeofcharge keeps its markers. The
length inside the plasma comes from `inside_at` on points along the lines.

In [ ]:
r_start = float(wall.rz(0.0, 0.0)[0]) - 0.02
tangency = np.linspace(4.6, 6.0, 8)
beta = np.arcsin(tangency / r_start)
direction = np.stack((-np.cos(beta), np.sin(beta), np.zeros_like(beta)), axis=1)
start = np.broadcast_to([r_start, 0.0, 0.0], direction.shape)
end = start + 14.0 * direction

entry = plasma.first_hit(start, end)
spot = wall.first_hit(start, end)
assert spot.hit.all() and spot.refined.all()

s = np.linspace(0.0, 14.0, 2801)
inside = plasma.inside_at(start[:, None, :] + s[None, :, None] * direction[:, None, :])
path = np.asarray(inside).sum(axis=1) * (s[1] - s[0])

print("R_t [m]  plasma entry [m]  in plasma [m]  wall hit [m]  phi [deg]  theta [deg]")
for i, r_t in enumerate(tangency):
    first = f"{entry.distance[i]:16.3f}" if entry.hit[i] else f"{'missed':>16}"
    print(
        f"{r_t:6.2f} {first} {path[i]:14.2f} {spot.distance[i]:13.3f}"
        f" {np.degrees(spot.phi[i]):10.1f} {np.degrees(spot.theta[i]):12.1f}"
    )

`SurfaceHits` also gives the outward normal at each hit (here the exact surface's) and
the triangle of the search (`face`). The angle between line and normal sets how the
shine-through power spreads on the wall.

In [ ]:
incidence = np.degrees(np.arccos(np.abs(np.sum(spot.normal * direction, axis=1))))
print("angle of incidence on the wall [deg]:", np.round(incidence, 1))

## Top view

The outlines of the plasma and the wall in the plane $z = 0$ (from their inside tests on a
grid), the lines, where they enter the plasma and where they hit the wall.

In [ ]:
x, y = np.meshgrid(np.linspace(-6.6, 6.6, 500), np.linspace(-6.6, 6.6, 500))
plane = np.stack((x, y, np.zeros_like(x)), axis=-1)
fig, ax = plt.subplots(figsize=(6, 6))
for surface, color in ((plasma, "C0"), (wall, "C1")):
    inside = np.asarray(surface.inside_at(plane), dtype=float)
    ax.contour(x, y, inside, levels=[0.5], colors=color)
for i in range(len(tangency)):
    ax.plot(
        [start[i, 0], spot.point[i, 0]], [start[i, 1], spot.point[i, 1]], "0.5", lw=0.8
    )
ax.scatter(
    *entry.point[entry.hit, :2].T, color="C2", s=14, zorder=3, label="plasma entry"
)
ax.scatter(*spot.point[:, :2].T, color="C3", s=14, zorder=3, label="wall hit")
ax.plot([], [], "C0", label="plasma boundary")
ax.plot([], [], "C1", label="wall (stand-in)")
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.legend(loc="center", fontsize=8)
ax.set_title("Lines from the outboard midplane, z = 0")
plt.show()

## In 3D

`show_3d` draws the triangulation; a coarser one keeps the figure light. The hits are drawn
with their normals.

In [ ]:
from simgeometry import FourierSurface

coarse = FourierSurface(wall.r_cos, wall.z_sin, wall.nfp, resolution=(24, 12))
ax = coarse.show_3d(
    display=False,
    opacity=0.15,
    show_bounds=False,
    points={
        "plasma entry": entry.point[entry.hit].T,
        "wall hit": {"points": spot.point.T, "vectors": spot.normal.T},
    },
)
ax.view_init(elev=40, azim=-60)
ax.set_title("Stand-in wall with the wall hits and their normals")
plt.show()

## Accuracy and speed

- Whether a segment hits is decided by the triangles: within `triangulation_error` (here
  about 4 mm) of the surface, a grazing line may hit the triangles but not the surface or the
  other way round. Raise `resolution` to shrink it, quadratically in the spacing.
- The hits are then exact for the Fourier surface: against the closed form of a circular
  torus, to $10^{-10}$ m (see the tests). With `refine=False` they stay on the triangles.
- The compiled search takes about 0.1–0.2 s for $10^5$ lines against $10^5$ triangles on one
  core; the Newton steps onto the surface cost a few seconds more for $10^5$ hits with the
  wall's several hundred modes. The inside test takes about 0.1 s for $10^6$ points.